# SentinelleCoop — Modèle de détection LBC/FT## Du problème métier au modèle déployable en caisse**Hackathon CIF/DigiCoop 2026 — thème 1 — branche `feature/moteur-ia`**---### Avertissement de méthode, à lire avant toutCe notebook ne cherche pas à démontrer qu'une IA fonctionne. Il cherche à **savoir si elle apportequelque chose**, et il rapporte le résultat que la mesure donne — y compris quand ce résultat estnégatif.Trois règles tenues d'un bout à l'autre :1. **Aucune étiquette n'est inventée.** Là où il n'existe pas de vérité terrain, aucun classifieur   supervisé n'est entraîné et aucune précision n'est annoncée.2. **Le test n'est jamais utilisé pour choisir quoi que ce soit** — ni seuil, ni hyperparamètre.3. **Toute conclusion est accompagnée de sa variance.** Un écart plus petit que le bruit   d'échantillonnage n'est pas un écart.

## 1. Le problème métierUne caisse du réseau CIF, à Dori ou à Banfora, doit à chaque opération de guichet :- vérifier que le client n'est pas une personne sanctionnée ou politiquement exposée ;- repérer des opérations inhabituelles au regard de son profil ;- le faire **hors ligne**, sur un poste modeste, sans expertise data science sur place.Le coût des erreurs n'est pas symétrique :| Erreur | Conséquence ||---|---|| **Faux négatif** — une personne sanctionnée passe | Manquement réglementaire, sanction de l'autorité, risque réputationnel réseau || **Faux positif** — un client légitime est bloqué | Temps d'analyste, client mécontent, et surtout **fatigue d'alerte** : au-delà d'un certain volume, les alertes ne sont plus traitées, ce qui est un échec de conformité à part entière |C'est cette asymétrie — et non une métrique académique — qui doit piloter les choix.

## 2. Deux problèmes analytiques distincts, et un seul a des étiquettesIl faut séparer ce qui est mélangé dans la plupart des présentations « IA + LBC/FT » :| | Problème A — rapprochement de noms | Problème B — opérations inhabituelles ||---|---|---|| Question | « Ce client est-il la même personne que cette entrée de liste ? » | « Cette activité s'écarte-t-elle du normal ? » || Étiquettes réelles disponibles | **Oui** : `data/variantes_noms_ao.csv`, 99 paires attestées | **Non** : aucun cas de blanchiment confirmé dans le périmètre || Approche légitime | Apprentissage **supervisé**, mesurable | Détection d'écart **non supervisée** || Piège à éviter | — | Entraîner un classifieur sur des étiquettes produites par nos propres règles → le modèle réapprend les règles, en moins lisible |> **Décision.** Problème A : on entraîne et on mesure. Problème B : détection d'écart non supervisée,> sans aucune métrique de précision annoncée, en complément — non en remplacement — des règles> typologiques de `verdicts.py`.

## 3. Critères de succès, définis avant de modéliser**Technique.** À rappel au moins égal à celui de la baseline, réduire les faux positifs. Le scoreretenu est un **coût** : `coût = c_FN × FN + 1 × FP`, avec `c_FN` le nombre de fausses alertesqu'on accepte de traiter pour éviter un manquement. On rapporte plusieurs valeurs de `c_FN`plutôt qu'une seule choisie après coup.**Métier.** Le modèle doit tourner hors ligne sur un poste de guichet, être explicable ligne à lignedevant un auditeur, et ne jamais prendre seul une décision réglementaire.**Critère de promotion.** Le modèle ne remplace la règle existante que si son avantage est**supérieur au bruit d'échantillonnage**, mesuré sur plusieurs graines.

In [ ]:
import csv, json, math, random, sys, timefrom collections import defaultdictfrom datetime import datetimefrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as plt# Racine du depot : adapter si le notebook est deplace.RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(RACINE))from sentinellecoop.ingest import charger_onufrom sentinellecoop.matcher import PoidsJetons, similarite_nom, SEUIL_INFORMATIFfrom sentinellecoop.phonetics import (code_nom, deplier, jaro_winkler, jetons,                                      similarite_jeton)plt.rcParams["figure.figsize"] = (11, 4)print("racine        :", RACINE)print("seuil courant :", SEUIL_INFORMATIF)

---# PARTIE A — Modèle 1 : rapprochement de noms (supervisé)## 4. Les données réelles`data/variantes_noms_ao.csv` contient 99 paires **attestées** nom/variante, collectées sur despatronymes et prénoms ouest-africains réels. C'est la seule source d'étiquettes vraies du projet.

In [ ]:
familles, categories = defaultdict(set), {}with open(RACINE/"data"/"variantes_noms_ao.csv", encoding="utf-8-sig", newline="") as f:    for r in csv.DictReader(f):        a, b = r["reference"].strip(), r["variante"].strip()        familles[a] |= {a, b}        categories[a] = r["categorie"]familles = {k: sorted(v) for k, v in familles.items()}print(f"identites distinctes (familles) : {len(familles)}")print(f"formes ecrites totales          : {sum(len(v) for v in familles.values())}")pd.Series([categories[k] for k in familles]).value_counts().to_frame("familles")

In [ ]:
# Exemples : une identite = toutes ses graphies attesteesfor k in ["Ouédraogo", "Diallo", "Mohamed", "El Hadj Salifou Cissé"]:    print(f"{k:24} -> {familles[k]}")

## 5. EDA — pourquoi le problème est difficileLe référentiel ONU réel (1011 entrées, ~3745 libellés distincts avec les alias) sert de source de**négatifs durs** : ce sont les faux positifs que le système rencontre réellement en production, pasdes paires artificielles faciles. C'est la différence méthodologique la plus importante de cenotebook.

In [ ]:
onu = charger_onu()LIB_ONU = sorted({x for e in onu for x in [e.nom, *e.alias]})POIDS = PoidsJetons([f for v in familles.values() for f in v] + LIB_ONU)print(f"{len(onu)} entrees ONU -> {len(LIB_ONU)} libelles (nom + alias)")

In [ ]:
# Distribution des scores : vraies variantes  vs  paires ONU sans rapportvraies = [similarite_nom(a, b, POIDS)          for v in familles.values() for i, a in enumerate(v) for b in v[i+1:]]rng = random.Random(0)sans_rapport = [similarite_nom(rng.choice(LIB_ONU), rng.choice(LIB_ONU), POIDS)                for _ in range(4000)]fig, ax = plt.subplots()ax.hist(sans_rapport, bins=60, alpha=.65, density=True, label="paires ONU sans rapport")ax.hist(vraies, bins=30, alpha=.65, density=True, label="variantes attestees (meme personne)")ax.axvline(SEUIL_INFORMATIF, color="crimson", ls="--", lw=2,           label=f"seuil production {SEUIL_INFORMATIF}")ax.set_xlabel("similarite ponderee"); ax.set_ylabel("densite")ax.set_title("Le seuil doit separer deux distributions qui se chevauchent")ax.legend(); plt.tight_layout(); plt.show()print(f"99e centile des paires sans rapport : {np.percentile(sans_rapport, 99):.3f}")print(f"5e centile des vraies variantes     : {np.percentile(vraies, 5):.3f}")

La fenêtre entre les deux distributions est **étroite**. C'est exactement ce que documente`data/scenarios.md` et ce qui a justifié de relever le seuil de 0.80 à 0.88. Toute la question estde savoir si un modèle exploite mieux cette zone qu'un seuil unique.

## 6. Construction des paires — et prévention des fuitesLe découpage se fait **par identité**, avant toute génération de paires : une famille de nomsappartient entièrement à `train`, `validation` ou `test`. Le référentiel ONU est partitionné de lamême façon, sinon une entrée ONU pourrait servir de négatif dans deux splits à la fois.Quatre familles de paires :| Type | Étiquette | Rôle ||---|---|---|| `variante_attestee` | 1 | vérité terrain brute || `compose_meme_personne` | 1 | nom complet, deux graphies (nom + prénom attestés) || `negatif_meme_patronyme` | 0 | deux Ouédraogo différents — le cas d'homonymie réel || `negatif_onu_reel` | 0 | **vrais faux positifs** issus du filtrage contre la liste ONU |

In [ ]:
patro = {k for k in familles if len(jetons(k)) == 1}PRENOMS = sorted(k for k in patro if categories[k] == "arabe")NOMS = sorted(k for k in patro if categories[k] in              ("translitteration", "palatalisation", "sifflante", "wolof"))RANG_SPLIT = {"train": 0, "validation": 1, "test": 2}   # deterministe : jamais hash()def construire_paires(seed):    r = random.Random(seed)    cles = sorted(familles); r.shuffle(cles); n = len(cles)    sm = {k: ("train" if i < int(n*.6) else "validation" if i < int(n*.8) else "test")          for i, k in enumerate(cles)}    ol = list(LIB_ONU); random.Random(seed+1).shuffle(ol); m = len(ol)    osp = {"train": ol[:int(m*.6)], "validation": ol[int(m*.6):int(m*.8)], "test": ol[int(m*.8):]}    def paires(split):        fam = sorted(x for x in familles if sm[x] == split); fs = set(fam)        pr = [p for p in PRENOMS if p in fs] or PRENOMS        nm = [x for x in NOMS if x in fs] or NOMS        rr = random.Random(seed*100 + RANG_SPLIT[split]); L = []        for ref in fam:                                   # positifs attestes            F = familles[ref]            for i in range(len(F)):                for j in range(i+1, len(F)):                    L.append((F[i], F[j], 1, "variante_attestee"))        for _ in range(len(fam)*8):                       # positifs composes            nf, pf = rr.choice(nm), rr.choice(pr)            a = f"{rr.choice(familles[nf])} {rr.choice(familles[pf])}"            b = f"{rr.choice(familles[nf])} {rr.choice(familles[pf])}"            if a != b: L.append((a, b, 1, "compose_meme_personne"))        for _ in range(len(fam)*8):                       # negatifs : meme patronyme            if len(pr) < 2: break            x = rr.choice(nm); p1, p2 = rr.sample(pr, 2)            L.append((f"{rr.choice(familles[x])} {rr.choice(familles[p1])}",                      f"{rr.choice(familles[x])} {rr.choice(familles[p2])}",                      0, "negatif_meme_patronyme"))        cl = set()                                        # negatifs : ONU reel        while len(cl) < 60:            cl.add(f"{rr.choice(familles[rr.choice(nm)])} {rr.choice(familles[rr.choice(pr)])}")        for c in sorted(cl):            cand = [(similarite_nom(c, l, POIDS), l) for l in osp[split]]            for s, l in sorted((x for x in cand if x[0] >= 0.70), reverse=True)[:4]:                L.append((c, l, 0, "negatif_onu_reel"))        return L    return {s: paires(s) for s in ("train", "validation", "test")}t0 = time.perf_counter()D = construire_paires(42)print(f"construit en {time.perf_counter()-t0:.0f}s\n")for s, v in D.items():    kinds = pd.Series([k for *_, k in v]).value_counts().to_dict()    print(f"{s:11}: {len(v):4} paires  ({sum(l for *_, l, _ in [(a,b,l,k) for a,b,l,k in v])} positifs)  {kinds}")

In [ ]:
# Verification de fuite EXPLICITE : aucune forme ecrite partagee entre splitsF = {s: {deplier(a) for a, b, _, _ in v} | {deplier(b) for a, b, _, _ in v}     for s, v in D.items()}print("formes communes train/test      :", len(F["train"] & F["test"]))print("formes communes validation/test :", len(F["validation"] & F["test"]))

> Une fuite résiduelle de 0 à 1 forme peut apparaître selon la graine : un nom de client généré> peut coïncider avec une forme d'un autre split. C'est mesuré et rapporté à chaque exécution,> jamais masqué.

## 7. Les variables explicativesToutes doivent être calculables **en Python pur** à l'exécution — c'est la contrainte edge.| Variable | Ce qu'elle capte ||---|---|| `sim_ponderee` | la baseline actuelle : similarité de jetons pondérée par la rareté || `sim_brute` | la même, sans pondération || `jaro_deplie` | similarité au niveau de la chaîne entière, sans découpage en jetons || `code_wape_egal` | les deux noms ont exactement le même code phonétique || `min_jeton` | **le maillon faible** : le jeton le moins bien apparié. Un nom peut avoir un score global élevé parce qu'un seul jeton correspond très bien || `ecart_nb_jetons` | différence de nombre de jetons || `rarete_min` | rareté du jeton le plus commun — « Ouédraogo » ne discrimine presque rien |

In [ ]:
FEATURES = ["sim_ponderee", "sim_brute", "jaro_deplie", "code_wape_egal",            "min_jeton", "ecart_nb_jetons", "rarete_min"]def features(a, b):    ja, jb = jetons(a), jetons(b)    if ja and jb:        mini = min(max(similarite_jeton(x, y) for y in jb) for x in ja)        rare = min(POIDS.poids(x) for x in ja + jb)    else:        mini, rare = 0.0, 1.0    return [similarite_nom(a, b, POIDS), similarite_nom(a, b, None),            jaro_winkler(deplier(a), deplier(b)),            1.0 if code_nom(a) == code_nom(b) else 0.0,            mini, abs(len(ja)-len(jb))/max(len(ja), len(jb), 1), rare]pd.DataFrame([features("Ouédraogo Salifou", "Wedraogo Salif"),              features("Ouédraogo Salifou", "Ouedraogo Ibrahim"),              features("Diallo Mamadou", "Jallo Mamadu")],             columns=FEATURES,             index=["meme personne", "homonyme (patronyme commun)", "meme personne"]).round(3)

## 8. Protocole d'évaluationBaseline = le seuil de production 0.88 sur `sim_ponderee` seul. Modèle = régression logistique surles 7 variables, descente de gradient, régularisation L2.Le seuil du modèle est choisi **sur la validation uniquement**. L'expérience complète est rejouéesur **5 graines** : c'est ce qui permet de distinguer un vrai gain d'une fluctuation.

In [ ]:
CFN, CFP = 20.0, 1.0SEEDS = (42, 7, 123, 2024, 31337)def confusion(y, pred):    tp=int(((y==1)&pred).sum()); fp=int(((y==0)&pred).sum())    fn=int(((y==1)&~pred).sum()); tn=int(((y==0)&~pred).sum())    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=tp/max(tp+fp,1),                rappel=tp/max(tp+fn,1), cout=CFN*fn+CFP*fp)def entrainer(X, Y):    mu, sd = X["train"].mean(0), X["train"].std(0)+1e-9    xt, yt = (X["train"]-mu)/sd, Y["train"]    w = np.zeros(len(FEATURES)); b = 0.0    for _ in range(6000):        p = 1/(1+np.exp(-np.clip(xt@w+b, -40, 40)))        w -= 0.3*(xt.T@(p-yt)/len(yt) + 0.001*w)        b -= 0.3*float((p-yt).mean())    return w, b, mu, sddef seuil_sur_validation(sc, y):    best, bc = 0.5, float("inf")    for t in np.unique(np.round(sc, 4)):        c = confusion(y, sc >= t)["cout"]        if c < bc: best, bc = float(t), c    return bestR = []for sd_ in SEEDS:    D = construire_paires(sd_)    X = {s: np.array([features(a, b) for a, b, _, _ in v]) for s, v in D.items()}    Y = {s: np.array([l for _, _, l, _ in v], dtype=float) for s, v in D.items()}    Fs = {s: {deplier(a) for a,b,_,_ in v} | {deplier(b) for a,b,_,_ in v} for s, v in D.items()}    w, b, mu, sdv = entrainer(X, Y)    sc = lambda x: 1/(1+np.exp(-np.clip(((x-mu)/sdv)@w+b, -40, 40)))    base = confusion(Y["test"], X["test"][:, 0] >= SEUIL_INFORMATIF)    st = seuil_sur_validation(sc(X["validation"]), Y["validation"])    mod = confusion(Y["test"], sc(X["test"]) >= st)    R.append(dict(graine=sd_, fuite=len(Fs["train"] & Fs["test"]),                  base=base, mod=mod, w=w, seuil=st))    print(f"graine {sd_:>5} | baseline fp={base['fp']:3d} fn={base['fn']:2d} cout={base['cout']:5.0f}"          f" | modele fp={mod['fp']:3d} fn={mod['fn']:2d} cout={mod['cout']:5.0f}")

In [ ]:
g = lambda src, k: np.array([r[src][k] for r in R], dtype=float)resume = pd.DataFrame({    "cout moyen":  [f"{g('base','cout').mean():.1f} ± {g('base','cout').std():.1f}",                    f"{g('mod','cout').mean():.1f} ± {g('mod','cout').std():.1f}"],    "faux positifs": [f"{g('base','fp').mean():.1f} ± {g('base','fp').std():.1f}",                      f"{g('mod','fp').mean():.1f} ± {g('mod','fp').std():.1f}"],    "faux negatifs": [f"{g('base','fn').mean():.1f} ± {g('base','fn').std():.1f}",                      f"{g('mod','fn').mean():.1f} ± {g('mod','fn').std():.1f}"],    "rappel":        [f"{g('base','rappel').mean():.3f}", f"{g('mod','rappel').mean():.3f}"],}, index=["baseline 0.88", "modele logistique"])display(resume)gagne = int((g('mod','cout') < g('base','cout')).sum())ecart = g('mod','cout').mean() - g('base','cout').mean()print(f"\nModele strictement meilleur sur {gagne}/{len(R)} graines")print(f"Ecart moyen de cout : {ecart:+.1f}")print(f"Ecart-type des couts : baseline {g('base','cout').std():.1f}, modele {g('mod','cout').std():.1f}")print(f"-> l'ecart vaut {abs(ecart)/g('base','cout').std()*100:.0f} % d'un seul ecart-type.")

### Matrice de confusionLes quatre cases, moyennées sur les 5 graines. Elles sont nommées par ce qu'elles signifient auguichet, pas par leurs initiales : c'est la lecture dont un responsable conformité a besoin.| | le moteur signale | le moteur ne signale pas ||---|---|---|| **la personne EST celle de la liste** | contrôle réussi | **une personne sanctionnée franchit le guichet** || **la personne N'EST PAS celle de la liste** | **client légitime bloqué, temps d'analyste** | contrôle réussi |Les deux cases en gras sont les seules qui coûtent quelque chose. C'est leur arbitrage — et nonl'exactitude globale — qui décide du seuil.

In [ ]:
def matrice(src):    c = {k: float(np.mean([r[src][k] for r in R])) for k in ("tp", "fp", "fn", "tn")}    return pd.DataFrame(        [[round(c["tp"], 1), round(c["fn"], 1)],         [round(c["fp"], 1), round(c["tn"], 1)]],        index=["Réalité : même personne", "Réalité : personnes différentes"],        columns=["Moteur : correspondance", "Moteur : pas de correspondance"])print("BASELINE 0.88 — moyenne sur 5 graines")display(matrice("base"))print("\nMODÈLE LOGISTIQUE — moyenne sur 5 graines")display(matrice("mod"))print("\nLecture métier :")for src, lab in (("base", "baseline 0.88"), ("mod", "modèle")):    fn = np.mean([r[src]["fn"] for r in R]); fp = np.mean([r[src]["fp"] for r in R])    tn = np.mean([r[src]["tn"] for r in R]); tp = np.mean([r[src]["tp"] for r in R])    print(f"  {lab:16} {fn:.1f} personne(s) sanctionnée(s) non détectée(s) sur {tp+fn:.0f}"          f"  |  {fp:.1f} client(s) légitime(s) bloqué(s) sur {fp+tn:.0f}")pos = np.mean([r["base"]["tp"] + r["base"]["fn"] for r in R])neg = np.mean([r["base"]["fp"] + r["base"]["tn"] for r in R])print(f"\nRéserve sur la prévalence : le jeu de test compte {pos:.0f} paires positives"      f" pour {neg:.0f} négatives, soit environ 1 pour {neg/pos:.0f}.")print("Au guichet, la prévalence réelle est bien plus extrême : la quasi-totalité des")print("filtrages ne correspond à personne. À cette prévalence, le nombre de faux positifs")print("par vraie détection serait nettement plus élevé pour les deux approches.")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))x = np.arange(len(R)); l = 0.35a1.bar(x-l/2, g('base','cout'), l, label="baseline 0.88")a1.bar(x+l/2, g('mod','cout'), l, label="modele")a1.set_xticks(x); a1.set_xticklabels([r["graine"] for r in R])a1.set_xlabel("graine"); a1.set_ylabel("cout"); a1.legend()a1.set_title("Le classement s'inverse selon la graine")W = np.array([r["w"] for r in R])ordre = np.argsort(-np.abs(W.mean(0)))a2.barh([FEATURES[i] for i in ordre][::-1], W.mean(0)[ordre][::-1],        xerr=W.std(0)[ordre][::-1], color="steelblue")a2.axvline(0, color="k", lw=.8); a2.set_title("Poids appris (moyenne ± ecart-type)")plt.tight_layout(); plt.show()

## 9. Verdict — et il est négatifLes chiffres mesurés :| | coût | faux positifs | faux négatifs | rappel ||---|---|---|---|---|| baseline 0.88 | 24,0 ± 20,7 | **8,0 ± 6,8** | 0,8 ± 0,7 | 0,992 || modèle logistique | 23,0 ± 15,0 | **19,0 ± 16,5** | 0,2 ± 0,4 | 0,998 |**Le modèle n'est pas promu.** Trois raisons, toutes lisibles dans le tableau :1. L'écart moyen de coût est de **−1,0** face à des écarts-types de 20,7 et 15,0 : il vaut environ   **5 % d'un seul écart-type**. C'est du bruit, pas un gain.2. Le modèle **plus que double les faux positifs** (8 → 19) pour éviter 0,6 faux négatif en moyenne.   En caisse, cela signifie plus de deux fois plus d'alertes à traiter — le chemin direct vers la   fatigue d'alerte.3. Il ne gagne que sur **2 graines sur 5**. Un composant réglementaire dont le classement dépend du   tirage n'est pas déployable.Le poids de `sim_ponderee` est instable (−0,909 ± 1,029 : l'écart-type dépasse la moyenne, le signelui-même change selon la graine) — signe classique de colinéarité et de données insuffisantes.> **Conclusion honnête : le seuil 0.88, calibré empiriquement par l'équipe le 5 septembre, est déjà> quasi optimal.** L'expérience ne l'a pas amélioré — elle l'a **validé**. C'est un résultat, pas un> échec : il évite de déployer un composant plus lourd, moins lisible et non supérieur.

## 10. Le ML comme outil de diagnosticLes poids les plus stables désignent `jaro_deplie` (+2,52 ± 0,21) et `min_jeton` (+2,42 ± 0,60) —deux signaux que la baseline n'exploite pas. On a donc testé une **règle explicable** :`sim_ponderee ≥ 0.88 ET min_jeton ≥ t`, avec `t` choisi sur la validation.Résultat mesuré (5 graines, `t` choisi sur la validation) :| | coût moyen | FP moyen | FN moyen ||---|---|---|---|| baseline 0.88 | 24,0 ± 20,7 | 8,0 | 0,8 || 0.88 + `min_jeton ≥ t` | 23,8 ± 21,0 | 7,8 | 0,8 |La règle donne un résultat **identique à la baseline sur 4 graines sur 5**, et supprime exactement**un** faux positif sur la cinquième. Les `t` retenus vont de **0,383 à 0,754** selon la graine.Elle ne nuit donc jamais — mais son apport (0,2 de coût moyen) vaut **1 % d'un écart-type**. Etl'écart de 0,383 à 0,754 dans le seuil sélectionné montre qu'il n'y a pas assez de donnéesétiquetées pour calibrer de façon fiable ne serait-ce qu'un seul seuil supplémentaire.**On ne l'ajoute donc pas.** Ajouter un paramètre non calibrable à un moteur réglementaire, pour ungain indiscernable du bruit, c'est de la complexité sans contrepartie.

In [ ]:
# Verification de la regle 'min_jeton' — les chiffres ci-dessus sont produits ici.I_SIM, I_MIN = FEATURES.index("sim_ponderee"), FEATURES.index("min_jeton")lignes_regle = []for sd_ in SEEDS:    D = construire_paires(sd_)    X = {s: np.array([features(a, b) for a, b, _, _ in v]) for s, v in D.items()}    Y = {s: np.array([l for _, _, l, _ in v], dtype=float) for s, v in D.items()}    base = confusion(Y["test"], X["test"][:, I_SIM] >= SEUIL_INFORMATIF)    bt, bc = 0.0, float("inf")    for t in np.unique(np.round(X["validation"][:, I_MIN], 3)):        c = confusion(Y["validation"], (X["validation"][:, I_SIM] >= SEUIL_INFORMATIF)                      & (X["validation"][:, I_MIN] >= t))["cout"]        if c < bc: bt, bc = float(t), c    reg = confusion(Y["test"], (X["test"][:, I_SIM] >= SEUIL_INFORMATIF)                    & (X["test"][:, I_MIN] >= bt))    lignes_regle.append({"graine": sd_, "cout baseline": base["cout"],                         "cout regle": reg["cout"], "t retenu": round(bt, 3),                         "identique ?": base["cout"] == reg["cout"]})df_regle = pd.DataFrame(lignes_regle)display(df_regle)print(f"identique a la baseline sur {int(df_regle['identique ?'].sum())}/{len(df_regle)} graines")print(f"cout moyen : baseline {df_regle['cout baseline'].mean():.1f}  "      f"regle {df_regle['cout regle'].mean():.1f}")

## 11. Ce qu'il faudrait pour conclure autrementL'obstacle n'est pas l'algorithme, c'est le volume d'étiquettes : **99 paires attestées**, soitenviron 30 identités par split.Pour trancher, il faudrait :- **de l'ordre de 1 000 à 5 000 paires étiquetées** par un analyste conformité, issues d'un  portefeuille réel anonymisé ;- les **décisions historiques d'analystes** sur les alertes passées (confirmée / levée), qui sont la  vraie vérité terrain de ce métier ;- des **négatifs durs réels** : les faux positifs effectivement rencontrés au guichet.C'est exactement l'échantillon anonymisé demandé au §4.2 de la note de présentation. Sans lui,ajouter un modèle serait de l'habillage.

---# PARTIE B — Modèle 2 : score de risque transactionnel (non supervisé)## 12. Pourquoi pas de classifieur supervisé iciIl n'existe aucun cas de blanchiment confirmé dans le périmètre. Les seules étiquettes qu'onpourrait fabriquer viendraient de nos propres règles (`verdicts.py`) — un modèle entraîné dessusréapprendrait ces règles, avec une couche d'opacité en plus et des modes d'échec en plus. Ce seraitmesurable, impressionnant, et **faux**.On fait donc de la **détection d'écart** : on ne prétend pas savoir ce qui est frauduleux, on mesurece qui s'écarte de la population, avec des statistiques robustes (médiane + MAD, insensibles auxvaleurs extrêmes contrairement à moyenne + écart-type).**Aucune précision ni rappel ne sera annoncé** — il n'y a rien contre quoi les mesurer.

In [ ]:
VARIABLES = ["nb_ops", "volume_total", "montant_max", "ratio_especes",             "ratio_nuit", "nb_contreparties", "ecart_type_montants", "ratio_entrees"]# Profils de comportement NORMAL. Ces parametres SONT la calibration : ils sont# documentes ici et doivent etre refaits sur un portefeuille reel anonymise.PROFILS = {    "salarie":      dict(ops=(6, 14),  moy=(60_000, 180_000), esp=.35, nuit=.02, ctp=(1, 3)),    "commercant":   dict(ops=(40, 90), moy=(80_000, 300_000), esp=.80, nuit=.06, ctp=(4, 15)),    "agriculteur":  dict(ops=(4, 12),  moy=(40_000, 150_000), esp=.90, nuit=.01, ctp=(1, 4)),    "transporteur": dict(ops=(20, 50), moy=(50_000, 200_000), esp=.70, nuit=.10, ctp=(3, 10)),}def population(n=300, seed=42):    r = random.Random(seed); out = []    for i in range(n):        prof = r.choice(list(PROFILS)); p = PROFILS[prof]        nb = r.randint(*p["ops"])        montants = [max(1000, r.gauss(r.uniform(*p["moy"]), r.uniform(*p["moy"])*.4))                    for _ in range(nb)]        out.append({"client_id": f"SYN-{i:04d}", "profil": prof, "nb_ops": float(nb),                    "volume_total": float(sum(montants)), "montant_max": float(max(montants)),                    "ratio_especes": min(1., max(0., r.gauss(p["esp"], .1))),                    "ratio_nuit": min(1., max(0., r.gauss(p["nuit"], .03))),                    "nb_contreparties": float(r.randint(*p["ctp"])),                    "ecart_type_montants": float(np.std(montants)),                    "ratio_entrees": min(1., max(0., r.gauss(.55, .15)))})    return outpop = population()pd.DataFrame(pop).groupby("profil")[VARIABLES].median().round(0)

In [ ]:
def calibrer(lignes):    cal = {}    for v in VARIABLES:        col = np.array([x[v] for x in lignes], float)        med = float(np.median(col))        mad = float(np.median(np.abs(col - med)))        cal[v] = {"mediane": med, "mad": mad if mad > 1e-9 else float(col.std() or 1.0)}    return calCAL = calibrer(pop)# Poids metier : CHOIX HUMAIN documente, pas appris — aucune etiquette ne# permettrait de les apprendre.POIDS_RISQUE = {"nb_ops": .8, "volume_total": 1.0, "montant_max": 1.2, "ratio_especes": 1.0,                "ratio_nuit": 1.3, "nb_contreparties": 1.1, "ecart_type_montants": .7,                "ratio_entrees": .5}def score_risque(c, cal=CAL, poids=POIDS_RISQUE):    '''Score d'ecart + contribution exacte de chaque variable.    Seules les deviations VERS LE HAUT comptent ; ecretage a 6 pour qu'une    variable seule ne sature pas le score.'''    contrib = {v: round(max(0., min(6., (float(c[v]) - cal[v]["mediane"])                                    / (1.4826*cal[v]["mad"]))) * poids[v], 3)               for v in VARIABLES}    return round(sum(contrib.values()), 3), dict(sorted(contrib.items(), key=lambda k: -k[1]))pd.DataFrame(CAL).T.round(2)

## 13. Vérification de bon sens sur les scénarios connusOn applique le score aux 3 clients du dataset de démonstration, dont les typologies sont documentéesdans `data/scenarios.md` :- **C-1029** : fractionnement + collecte FT + activation-dispersion FT- **C-3091** : PPE + compte rebond- **C-2214** : témoin neutre, aucune alerte attendue

In [ ]:
comptes = {}with open(RACINE/"data"/"comptes.csv", encoding="utf-8-sig", newline="") as f:    for r in csv.DictReader(f): comptes[r["id"]] = r["client_id"]tx = defaultdict(list)with open(RACINE/"data"/"transactions.csv", encoding="utf-8-sig", newline="") as f:    for r in csv.DictReader(f):        if comptes.get(r["compte_id"]): tx[comptes[r["compte_id"]]].append(r)demo = []for cid, ops in tx.items():    mts = [float(o["montant"]) for o in ops]    hrs = [datetime.strptime(o["date_heure"], "%Y-%m-%d %H:%M").hour for o in ops]    demo.append({"client_id": cid, "nb_ops": float(len(ops)), "volume_total": float(sum(mts)),                 "montant_max": float(max(mts)),                 "ratio_especes": sum(1 for o in ops if o["canal"] == "guichet")/len(ops),                 "ratio_nuit": sum(1 for h in hrs if h >= 22 or h <= 5)/len(ops),                 "nb_contreparties": float(len({o["compte_contrepartie_id"] for o in ops                                                if o["compte_contrepartie_id"]})),                 "ecart_type_montants": float(np.std(mts)),                 "ratio_entrees": sum(1 for o in ops if o["sens"] == "entree")/len(ops)})scores_pop = np.array([score_risque(c)[0] for c in pop])lignes = []for c in sorted(demo, key=lambda x: -score_risque(x)[0]):    s, contrib = score_risque(c)    top = ", ".join(f"{k} (+{v})" for k, v in list(contrib.items())[:3] if v > 0) or "aucune"    lignes.append({"client": c["client_id"], "score": s,                   "centile population": round(float((scores_pop < s).mean()*100), 1),                   "principales contributions": top})pd.DataFrame(lignes)

In [ ]:
fig, ax = plt.subplots()ax.hist(scores_pop, bins=40, alpha=.75, label="population synthetique (n=300)")for c in demo:    s, _ = score_risque(c)    ax.axvline(s, ls="--", lw=2, label=f"{c['client_id']} = {s:.1f}")ax.set_xlabel("score d'ecart"); ax.set_ylabel("nombre de clients")ax.set_title("Position des scenarios connus dans la population")ax.legend(); plt.tight_layout(); plt.show()

**Lecture honnête de ce résultat.** Le témoin C-2214 obtient exactement 0 et les deux clients àtypologie obtiennent les scores les plus élevés des trois : l'ordre est correct. Mais ils nedépassent pas le 80e centile de la population — des commerçants légitimes à fort volume scorentaussi haut.Cela dit précisément ce que ce score est : **un outil de priorisation, pas de qualification.** Ilordonne une file d'attente d'analyse. C'est `verdicts.py` qui qualifie une typologie, avec une règlecitable devant un auditeur. Les deux sont complémentaires, et présenter ce score comme un détecteurde fraude serait mentir.

---# PARTIE C — Intégration edge## 14. Ce qui part réellement en caisseLe point clé de l'argument « léger » : **numpy, pandas et matplotlib ne servent qu'ici, dans cenotebook.** Ils ne sont jamais installés sur le poste de guichet.```   CE NOTEBOOK (portable, hors ligne, une fois)        LE POSTE DE GUICHET   numpy / pandas / matplotlib                          Python standard seul   ──────────────────────────────────────►  artefact JSON  ──────────────────►   entrainement, mesure, calibration        quelques centaines    produit scalaire                                            d'octets              + sigmoide```C'est la même relation qu'entre un compilateur et le binaire qu'il produit : l'outil deconstruction n'est pas une dépendance d'exécution.

In [ ]:
# Inference en Python PUR — aucune dependance. C'est ce code qui tourne en caisse.def predire_pur(x, modele):    z = modele["biais"] + sum(a*b for a, b in zip(x, modele["poids"]))    return 1/(1 + math.exp(-max(-40, min(40, z))))def score_risque_pur(client, artefact):    tot, contrib = 0.0, {}    for v in artefact["variables"]:        c = artefact["calibration"][v]        z = (float(client[v]) - c["mediane"]) / (1.4826*c["mad"])        contrib[v] = round(max(0., min(artefact["ecretage_z"], z)) * artefact["poids"][v], 3)        tot += contrib[v]    return round(tot, 3), contribartefact_risque = {"version": "ecart-robuste-v1", "type": "non_supervise",                   "variables": VARIABLES, "calibration": CAL, "poids": POIDS_RISQUE,                   "ecretage_z": 6.0,                   "statut": "CALIBRATION_SYNTHETIQUE_A_REFAIRE_SUR_DONNEES_REELLES"}# Verification : le Python pur donne le meme resultat que la version numpyfor c in demo:    a, _ = score_risque(c); b, _ = score_risque_pur(c, artefact_risque)    assert abs(a-b) < 1e-9, (c["client_id"], a, b)print("Python pur == version numpy : OK sur les 3 clients")t0 = time.perf_counter()for _ in range(10_000):    score_risque_pur(demo[0], artefact_risque)print(f"latence inference : {(time.perf_counter()-t0)/10_000*1e6:.1f} microsecondes par client")blob = json.dumps(artefact_risque, ensure_ascii=False)print(f"taille de l'artefact : {len(blob.encode()):,} octets "      f"({len(blob.encode())/512/1024*100:.2f} % du budget 512 Ko)")

In [ ]:
# Exemple de sortie destinee au guichet : chaque point de score est attribuable.s, contrib = score_risque_pur(demo[0], artefact_risque)sortie = {"client_id": demo[0]["client_id"], "score_ecart": s,          "type_de_score": "PRIORISATION_NON_CALIBREE",          "contributions": {k: v for k, v in sorted(contrib.items(), key=lambda x: -x[1]) if v > 0},          "decision_operationnelle": "NON_PRISE_PAR_LE_MOTEUR",          "action_recommandee": "REVUE_ANALYSTE"}print(json.dumps(sortie, indent=2, ensure_ascii=False))

## 15. Où cela se branche| Composant | Ce qu'il reçoit ||---|---|| `sentinellecoop/screen.py` | filtrage nominal — **inchangé**, le seuil 0.88 est confirmé par la partie A || `sentinellecoop/verdicts.py` | typologies LBC/FT — **inchangé**, reste la source des qualifications || `sentinellecoop-app/backend` | expose le score d'écart en complément des verdicts || `guichet/moteur.js` | le score d'écart est portable en JS : ce sont des soustractions et des divisions |**Rien de ce notebook ne remplace un composant existant.** La partie A a validé la baseline, lapartie B ajoute une priorisation. C'est un ajout non régressif.

## 16. Surveillance et réentraînementUn modèle peut devenir faux sans qu'une ligne de code change.| À surveiller | Signal | Action ||---|---|---|| Dérive des données | la médiane d'une variable s'éloigne de la calibration | recalibrer médianes/MAD || Volume d'alertes | plus d'alertes que la caisse ne peut en traiter | revoir les poids métier, pas le seuil en douce || Référentiel périmé | dernière synchronisation > 7 jours | état indéterminé, politique de vigilance renforcée || Nouvelles typologies | un cas confirmé qu'aucune règle n'attrape | ajouter une règle dans `verdicts.py` |La calibration porte une date et une version. Toute modification de seuil doit être tracée dans`CHANGELOG.md` avec sa justification — c'est déjà la discipline du dépôt.

## 17. Limites — à énoncer devant le jury, pas à cacher1. **99 paires attestées.** Insuffisant pour démontrer qu'un modèle appris bat le seuil actuel.   L'expérience conclut à l'absence de gain, pas à un gain.2. **La calibration du score d'écart est synthétique.** Médianes et MAD viennent d'une population   générée avec les paramètres documentés en partie B, pas d'un portefeuille réel.3. **Les poids métier du score d'écart sont un choix humain**, pas un résultat d'apprentissage.4. **Aucune vérité terrain pour le score transactionnel** : il priorise, il ne qualifie pas.5. **Le score n'est pas calibré en probabilité.** Un score de 8 ne signifie pas « 8 % de risque ».6. **Aucune décision réglementaire n'est prise par le moteur.** Chaque sortie porte   `decision_operationnelle = NON_PRISE_PAR_LE_MOTEUR`.> Ce qui est livré : un protocole d'évaluation reproductible, la validation mesurée du seuil> existant, un score de priorisation explicable ligne à ligne, et un chemin d'exécution sans aucune> dépendance sur le poste de guichet.